# Novel Hybrid CNN + ViT Research Ideas for APTOS 2019

Goal: generate paper-level architectural innovations for diabetic retinopathy classification using an 80/10/10 split (train/validation/test).

All ideas below avoid implementation details and focus on design rationale, mathematical formulation, and research novelty.

## [Idea 1] Confidence-Gated Local-Global Fusion (Simple)

- Problem: Baseline CNN+ViT fusion often uses static concatenation, which mixes noisy CNN local cues and unstable ViT global tokens equally.
- Proposed Change: Add a confidence-gated fusion block that learns per-sample weights for CNN and ViT streams.
- Technical Description: Let local feature be $f_c$ and global feature be $f_t$. Compute branch confidences $s_c, s_t n [0,1]$ from lightweight heads, then fuse as $f = lpha f_c + (1-lpha) f_t$ where $lpha = igma(w^	op[s_c, s_t])$. Add entropy regularization to prevent gate collapse.
- Why It’s Novel: Instead of feature-only gating, the gate is confidence-conditioned, making fusion uncertainty-aware at feature level.
- Expected Benefit: Better robustness on hard/ambiguous retinal images, reduced overfitting to one branch, modest accuracy gain with minimal complexity.

## [Idea 2] Lesion-Scale Spatial-Channel Co-Attention Pyramid

- Problem: DR lesions vary strongly in size (microaneurysms to hemorrhages), while single-scale attention misses fine-to-coarse patterns.
- Proposed Change: Introduce a multi-scale co-attention pyramid combining channel attention (what) and spatial attention (where) at each scale before ViT tokenization.
- Technical Description: Build feature pyramid levels $P_1,P_2,P_3$. For each level, compute channel attention $A_c=	ext{MLP}(	ext{GAP}(P_i))$ and spatial attention $A_s=	ext{Conv}(	ext{Pool}(P_i))$. Recalibrate as $at P_i=P_idot A_cdot A_s$. Feed recalibrated maps to patch embedding with scale tags.
- Why It’s Novel: Joint scale-specific spatial-channel recalibration tightly coupled to token generation, not post-hoc attention.
- Expected Benefit: Improved sensitivity to subtle lesions, better class-wise recall for mild/moderate DR, stronger interpretability via scale-attention maps.

## [Idea 3] Uncertainty-Driven Token Refinement Transformer

- Problem: ViT processes many irrelevant patches; noisy retinal background tokens can dominate attention and increase overfitting.
- Proposed Change: Add uncertainty-aware token refinement that iteratively prunes or reweights uncertain tokens.
- Technical Description: For token $t_i$, estimate predictive uncertainty $u_i$ via evidential head or MC-dropout proxy. Define retention score $r_i=xp(-eta u_i)$. Perform soft pruning by $t'_i=r_i t_i$ and hard top-k keep in later blocks. Add refinement consistency loss across layers to stabilize token set evolution.
- Why It’s Novel: Token selection is driven by uncertainty signals rather than only attention magnitude, aligning representation quality with calibration.
- Expected Benefit: Better generalization, less background overfitting, lower computation in deeper transformer stages.

## [Idea 4] Ordinal-Distribution Aware QWK Optimization

- Problem: DR grades are ordinal (0 to 4), but standard cross-entropy treats classes as independent categories, harming near-miss behavior.
- Proposed Change: Use a hybrid objective combining ordinal distribution learning, differentiable QWK surrogate, and class-frequency adaptive margin.
- Technical Description: Predict class distribution $p$ and cumulative logits for ordinal thresholds. Minimize $athcal{L}=ambda_1athcal{L}_{ord}+ambda_2athcal{L}_{qwk}+ambda_3athcal{L}_{bal}$. Set class margin $m_c=amma/qrt{n_c}$ for minority classes. Include distance-aware penalty $d(y,at y)^2$ to punish far ordinal errors more.
- Why It’s Novel: Integrates ordinal geometry, imbalance adaptation, and metric-aligned optimization in one mathematically coherent objective.
- Expected Benefit: Improved QWK and clinically meaningful error profile (fewer severe misclassifications), stronger minority-class performance.

## [Idea 5] Dual-Stream Cross-Attention with Lesion Prototype Memory

- Problem: CNN and ViT branches may learn redundant features and fail to preserve class-discriminative lesion semantics.
- Proposed Change: Add prototype memory bank per DR grade and force both streams to align through cross-attention to shared prototypes.
- Technical Description: Maintain prototypes $M_c$ updated by momentum. CNN queries ViT keys/values and vice versa, then both attend to $M_c$. Add prototype compactness/separation losses: $athcal{L}_{proto}=um ||z-M_y||_2^2-taum_{c
eq y}||z-M_c||_2^2$.
- Why It’s Novel: Unifies branch fusion and class-structure learning through explicit memory-guided relation modeling.
- Expected Benefit: Better inter-class separability, more stable fusion, improved interpretability via prototype-nearest lesion patterns.

## [Idea 6] Topology-Aware Retinal Graph Transformer (Structural)

- Problem: Flat token attention ignores anatomical relations (vessel connectivity, macula-disc context), which are crucial in DR assessment.
- Proposed Change: Construct a graph over lesion candidates and anatomical regions, then fuse graph embeddings with ViT tokens.
- Technical Description: Nodes represent lesion patches and anatomical anchors; edges encode spatial distance, vessel-path affinity, and co-occurrence prior. Run graph attention network to get $g_i$, then inject into transformer via relation bias $B_{ij}$ in attention: $	ext{Attn}(Q,K,V)=	ext{softmax}(QK^T/qrt{d}+B)V$.
- Why It’s Novel: Moves from image-grid-only reasoning to relation-aware clinical topology modeling inside hybrid CNN+ViT.
- Expected Benefit: Better structural consistency, stronger performance on subtle pathological patterns, improved clinician trust from relation maps.

## [Idea 7] Frequency-Spatial Dual Domain Hybrid Encoder

- Problem: Spatial-only encoders can miss texture-frequency clues (exudate edges, vessel irregularity) important for early DR.
- Proposed Change: Add a parallel frequency branch (e.g., wavelet or FFT patches) and perform cross-domain attention with spatial tokens.
- Technical Description: Decompose image into multi-band coefficients $F_b$. Encode $F_b$ with lightweight CNN/transformer and perform dual attention: spatial queries frequency keys, and frequency queries spatial keys. Adaptive gate learns per-sample domain importance.
- Why It’s Novel: Explicitly models spectral pathology cues and their interaction with anatomical spatial cues in one hybrid framework.
- Expected Benefit: Better early-stage detection, improved generalization under illumination/contrast variation, less overfit to color artifacts.

## [Idea 8] Mixture-of-Experts Severity Router with Progressive Fusion (Complex)

- Problem: A single shared backbone underfits diverse lesion morphologies across severity levels.
- Proposed Change: Build expert sub-networks specialized for low/mid/high severity and route samples dynamically using a calibrated router.
- Technical Description: Shared stem -> router predicts expert weights $i_k$. Each expert has distinct CNN+ViT fusion style (local-heavy, balanced, global-heavy). Final representation: $z=um_k i_k z_k$. Add load-balancing regularizer and uncertainty-temperature routing to avoid expert collapse.
- Why It’s Novel: Severity-aware specialization integrated with hybrid fusion strategy diversity, not just generic MoE replacement.
- Expected Benefit: Improved accuracy across all grades, better tail-class handling, modular interpretability through expert activation patterns.

## [Idea 9] Causal Counterfactual Lesion Consistency Hybrid

- Problem: Model may exploit confounders (illumination, camera artifacts) instead of causal lesion evidence.
- Proposed Change: Introduce lesion-focused counterfactual consistency by perturbing non-lesion regions and enforcing prediction stability.
- Technical Description: Generate counterfactual views by masking or style-shifting background while preserving lesion candidates. Train with causal consistency loss $athcal{L}_{cc}=D_{KL}(p(y|x)p(y|x_{cf}))$ plus lesion-attention alignment between original/counterfactual. Couple this with CNN saliency and ViT attention agreement loss.
- Why It’s Novel: Embeds causal invariance directly into hybrid branch interaction and attention alignment, beyond standard augmentation.
- Expected Benefit: Stronger out-of-distribution robustness, reduced shortcut learning, improved reliability in real clinical deployment.

## [Idea 10] Tri-Level Uncertainty-Calibrated Self-Distillation Hybrid (Publication-Level)

- Problem: Deep hybrids overfit small medical datasets and produce overconfident predictions despite good validation scores.
- Proposed Change: Design tri-level self-distillation (feature-level, token-level, logit-level) with uncertainty-weighted consistency and ordinal calibration head.
- Technical Description: Teacher is EMA of student. Distill CNN intermediate maps, ViT token relations (attention matrices), and output distributions. Weight each distillation term by confidence reliability $w=1-at u$. Add calibration head trained with Brier+ECE surrogate and ordinal monotonicity constraints.
- Why It’s Novel: Jointly distills cross-branch internal structure and calibrates uncertainty under ordinal constraints, forming an end-to-end clinically aware training paradigm.
- Expected Benefit: State-of-the-art generalization on APTOS split, better confidence quality, fewer catastrophic high-grade misses, high paper novelty potential.

## Suggested Research Bundles

1. Lightweight strong baseline: Idea 1 + Idea 2 + Idea 4
2. Mid-complexity high gain: Idea 3 + Idea 5 + Idea 7
3. Publication-oriented full framework: Idea 6 + Idea 8 + Idea 9 + Idea 10

These bundles allow progressive experimentation from simple ablations to full paper-level contributions.